<!-- NB_DOC_INTRO_V1 -->
# Python avance — cheat-sheet pour data scientists

> 📚 **Doc thematique** : [docs/01_STRUCTURES.md](docs/01_STRUCTURES.md) (Structures Python / DataFrame / Preprocessing)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Cheat-sheet des **particularites du langage Python** qui reviennent souvent en data science : fonctions d'ordre superieur, introspection, f-strings modernes, decorators (`functools`), context managers, generateurs (`yield`), pattern matching (3.10+), type hints avances. Tout code execute sur Python 3.11+.

Pour le contenu pandas / numpy : voir [Structures_L_T_D_Cheat_Sheet.ipynb](./Structures_L_T_D_Cheat_Sheet.ipynb) et [Structures_DataFrame.ipynb](./Structures_DataFrame.ipynb).

## Plan

1. Fonctions d'ordre superieur (map, filter, lambda, partial, reduce)
2. f-strings avancees (debug `=`, padding, format spec)
3. Decorators (timer, cache, retry)
4. Generateurs (`yield`, comprehensions)
5. Context managers (`with`, `contextlib`)
6. Type hints avances (Protocol, TypedDict, Literal, Generic)
7. Pattern matching (`match/case`, 3.10+)
8. Introspection (__code__, inspect)
9. Pieges et anti-patterns
10. References


## 1. Fonctions d'ordre superieur

Des fonctions qui prennent ou renvoient des fonctions. Base du paradigme fonctionnel en Python.

In [ ]:
from functools import partial, reduce, lru_cache
from operator import add, mul

# === map / filter / lambda ===
nums = [1, 2, 3, 4, 5]
print("map  squares :", list(map(lambda x: x**2, nums)))
print("filter evens :", list(filter(lambda x: x % 2 == 0, nums)))

# === reduce (somme accumulee) ===
print("reduce sum   :", reduce(add, nums))
print("reduce prod  :", reduce(mul, nums))

# === partial : pre-remplir des arguments ===
def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube = partial(power, exp=3)
print(f"square(5)={square(5)}, cube(5)={cube(5)}")

# === Closure (fonction qui capture des vars de son scope) ===
def make_multiplier(factor):
    def multiplier(x):
        return x * factor
    return multiplier
times3 = make_multiplier(3)
print(f"times3(10) = {times3(10)}")

## 2. f-strings — formatage moderne

Python 3.6+, ameliore en 3.8 (`=`) et 3.12 (multi-line). LE format a utiliser par defaut.

In [ ]:
x = 3.14159
y = 42
name = "Alice"

# Basique
print(f"x = {x}, name = {name}")

# Format spec (apres ':')
print(f"x rounded 2  : {x:.2f}")
print(f"x as percent : {x:.1%}")
print(f"y padded     : {y:>5}")        # right-align width 5
print(f"y zero-pad   : {y:05d}")
print(f"y hex        : {y:#x}")
print(f"y binary     : {y:b}")

# Debug equals (3.8+) — affiche l'expression + sa valeur
print(f"{x=}, {y=}")

# Expressions dans les f-strings
nums = [1, 2, 3]
print(f"sum = {sum(nums)}, max = {max(nums)}, n = {len(nums)}")

# Nested format
width = 10
print(f"{x:>{width}.2f}")

# Multi-line (3.12+)
big = f"""
Resume :
  x = {x:.2f}
  y = {y}
  name = {name}
"""
print(big)

## 3. Decorators — composer du comportement

Un decorator est une fonction qui prend une fonction et retourne une fonction (souvent decorée).

In [ ]:
import time
from functools import wraps

# === Decorator timer ===
def timer(func):
    @wraps(func)  # preserve nom + docstring de l'original
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        print(f"  [timer] {func.__name__} took {elapsed*1000:.2f} ms")
        return result
    return wrapper

@timer
def slow_func(n):
    """Boucle inutile pour la demo."""
    return sum(i**2 for i in range(n))

result = slow_func(100_000)
print(f"Result : {result}")
print(f"Name preserve : {slow_func.__name__}")
print(f"Doc preserve  : {slow_func.__doc__!r}")

In [ ]:
# === Decorator avec arguments (decorator factory) ===
def retry(max_attempts=3, delay=0.1):
    """Retry une fonction jusqu'a max_attempts en cas d'exception."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"  [retry] attempt {attempt} failed ({e}), retrying...")
                    time.sleep(delay)
        return wrapper
    return decorator

attempts = [0]
@retry(max_attempts=3, delay=0.01)
def flaky():
    attempts[0] += 1
    if attempts[0] < 3:
        raise ValueError(f"simulated failure #{attempts[0]}")
    return f"OK after {attempts[0]} attempts"

print(flaky())

# === lru_cache — memoisation built-in ===
@lru_cache(maxsize=128)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(f"fib(50) = {fib(50)}")
print(f"Cache info : {fib.cache_info()}")

## 4. Generateurs — economie memoire avec `yield`

Un generateur produit ses valeurs **a la demande** (lazy). Memoire constante meme sur des sequences infinies.

In [ ]:
# === Generator function ===
def fibonacci_gen():
    a, b = 0, 1
    while True:                  # infini !
        yield a
        a, b = b, a + b

gen = fibonacci_gen()
first_10 = [next(gen) for _ in range(10)]
print(f"10 premiers fibonacci : {first_10}")

# === Generator expression (comprehension lazy) ===
squares_gen = (x**2 for x in range(1_000_000))
print(f"Type : {type(squares_gen).__name__}")    # generator (vs list)
print(f"Sum  : {sum(squares_gen)}")              # ne materialise pas la liste

# Comparer memoire
import sys
list_squares = [x**2 for x in range(10_000)]
gen_squares = (x**2 for x in range(10_000))
print(f"List size : {sys.getsizeof(list_squares):,} bytes")
print(f"Gen size  : {sys.getsizeof(gen_squares):,} bytes  (constant !)")

# === yield from (delegation de generateur) ===
def chain(*iterables):
    for it in iterables:
        yield from it

print(list(chain([1, 2, 3], (4, 5), {6, 7})))

## 5. Context managers — `with` et `contextlib`

Garantit le cleanup (fichier ferme, lock relache) meme en cas d'exception.

In [ ]:
from contextlib import contextmanager
import tempfile, os

# === Built-in : open() est un context manager ===
tmp = os.path.join(tempfile.gettempdir(), "demo.txt")
with open(tmp, "w") as f:
    f.write("Hello")
# f est garanti ferme ici, meme si une exception arrive

# === Custom via @contextmanager ===
@contextmanager
def timer_block(label):
    t0 = time.perf_counter()
    yield                              # le code dans le with s'execute ici
    elapsed = time.perf_counter() - t0
    print(f"[{label}] {elapsed*1000:.2f} ms")

with timer_block("compute"):
    total = sum(i**2 for i in range(100_000))

# === suppress : ignorer une exception specifique ===
from contextlib import suppress
with suppress(FileNotFoundError):
    os.remove("/nonexistent/file.txt")    # n'erreur pas

# Cleanup
os.remove(tmp)
print("Cleanup OK")

## 6. Type hints avances — `Protocol`, `TypedDict`, `Literal`

Python 3.8+. Plus expressifs que les hints basiques.

In [ ]:
from typing import Protocol, TypedDict, Literal, Optional, Union, Generic, TypeVar

# === Protocol : duck typing structurel ===
class HasArea(Protocol):
    def area(self) -> float: ...

class Square:
    def __init__(self, side: float):
        self.side = side
    def area(self) -> float:
        return self.side ** 2

def total_area(shapes: list[HasArea]) -> float:
    return sum(s.area() for s in shapes)

print(f"Total : {total_area([Square(2), Square(3)])}")   # OK : Square implements HasArea

# === TypedDict : dict avec schema ===
class UserDict(TypedDict):
    name: str
    age: int
    email: Optional[str]

u: UserDict = {"name": "Alice", "age": 30, "email": "a@b.c"}
print(f"User : {u}")

# === Literal : valeurs restreintes ===
def set_level(level: Literal["debug", "info", "warning", "error"]) -> None:
    print(f"Level set to {level}")

set_level("info")
# set_level("foo")  -> mypy erreur (mais runtime OK car pas enforce)

# === Generic[T] : classes generiques ===
T = TypeVar("T")
class Stack(Generic[T]):
    def __init__(self):
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        return self._items.pop()

s: Stack[int] = Stack()
s.push(1); s.push(2)
print(f"pop : {s.pop()}")

## 7. Pattern matching — `match/case` (Python 3.10+)

Switch/case Python. Bien plus puissant que dans d'autres langages : destructuring, guards, classes.

In [ ]:
# === Basique ===
def classify(value):
    match value:
        case 0:
            return "zero"
        case n if n > 0:                # guard
            return "positive"
        case n if n < 0:
            return "negative"
        case _:                          # default
            return "unknown"

for v in [0, 5, -3, 0.0]:
    print(f"{v} -> {classify(v)}")

# === Destructuring de tuples ===
def describe_point(p):
    match p:
        case (0, 0):
            return "origin"
        case (0, y):
            return f"on Y axis at y={y}"
        case (x, 0):
            return f"on X axis at x={x}"
        case (x, y):
            return f"at ({x}, {y})"
        case _:
            return "not a 2D point"

for p in [(0, 0), (0, 5), (3, 0), (1, 2), (1, 2, 3)]:
    print(f"{p} -> {describe_point(p)}")

# === Destructuring de classes ===
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

def handle(p):
    match p:
        case Point(x=0, y=0):
            return "origin"
        case Point(x=x, y=y) if x == y:
            return f"on diagonal at {x}"
        case Point():
            return f"some point"
    return "not a point"

print(handle(Point(0, 0)))
print(handle(Point(3, 3)))
print(handle(Point(1, 2)))

## 8. Introspection — `__code__`, `inspect`

Meta-programmation : inspecter le code Python lui-meme.

In [ ]:
import inspect

def example(a: int, b: str = "default", *args, **kwargs) -> bool:
    """Docstring example."""
    return True

# === Via __code__ (bas niveau) ===
co = example.__code__
print(f"Name           : {co.co_name}")
print(f"Filename       : {co.co_filename.split('/')[-1]}")
print(f"# args         : {co.co_argcount}")
print(f"Local vars     : {co.co_varnames}")

# === Via inspect (haut niveau, recommande) ===
sig = inspect.signature(example)
print(f"\nSignature : {sig}")
for name, param in sig.parameters.items():
    print(f"  {name}: kind={param.kind.name}, default={param.default}, annotation={param.annotation}")

print(f"\nReturn annotation : {sig.return_annotation}")
print(f"Source code first line : {inspect.getsource(example).splitlines()[0]}")

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| `lambda x: x` au lieu de `operator.itemgetter` / `attrgetter` | Plus rapide et lisible |
| Decorator sans `@wraps(func)` | Perd `__name__` / `__doc__` |
| `lru_cache(maxsize=None)` sur fonction avec args mutables | Memory leak |
| `default = []` dans args | Partage entre appels (mutable default trap) |
| `except:` nu | Catche tout incluant `KeyboardInterrupt` — utiliser `except Exception:` |
| `isinstance(x, dict)` au lieu de `Mapping` | Trop strict, dict-like genere rejetes |
| `os.path.join` au lieu de `pathlib.Path` | Path moderne, OO |
| Encoder sans `encoding="utf-8"` | Defaut depend de l'OS, casse sur Windows |
| Pas de `if __name__ == '__main__':` sur scripts | Module non importable proprement |
| `print` au lieu de `logging` | Pas de niveaux, pas de timestamp |


## References

### Documentation officielle
- Python docs : https://docs.python.org/3/
- `functools` : https://docs.python.org/3/library/functools.html
- `contextlib` : https://docs.python.org/3/library/contextlib.html
- `typing` : https://docs.python.org/3/library/typing.html
- PEP 634 (pattern matching) : https://peps.python.org/pep-0634/

### Voir aussi (collection)
- [Structures_L_T_D_Cheat_Sheet.ipynb](Structures_L_T_D_Cheat_Sheet.ipynb)
- [Structures_DataFrame.ipynb](Structures_DataFrame.ipynb)
- [Structures_Polars.ipynb](Structures_Polars.ipynb)
- [SoftEng_Tests_Quality_ML.ipynb](SoftEng_Tests_Quality_ML.ipynb)
